In [7]:
import os
import logging
import glob
import fastf1
import pandas as pd

Configuration

In [ ]:
YEARS = [2022,2023, 2024, 2025]
SESSION_TYPE = 'R'  # Race
CACHE_PATH = './Raw/f1_cache'
SCHEDULE_CACHE_DIR = './Raw/schedules'
OUTPUT_DIR = './Raw/Data/Processed'
INDIVIDUAL_DIR = os.path.join(OUTPUT_DIR, 'by_race')
MASTER_FILE = os.path.join(OUTPUT_DIR, 'all_races_2022_2025.csv')
 
os.makedirs(CACHE_PATH, exist_ok=True)
os.makedirs(SCHEDULE_CACHE_DIR, exist_ok=True)
os.makedirs(INDIVIDUAL_DIR, exist_ok=True)
fastf1.Cache.enable_cache(CACHE_PATH)
 
 
class RateLimitError(Exception):
    """Levée quand l'API distante renvoie une erreur de quota (ex: '500 calls/h')."""
    pass
 
 
def _is_rate_limit_error(e: Exception) -> bool:
    msg = str(e).lower()
    return 'calls/h' in msg or '429' in msg or 'rate limit' in msg
 
 
def get_schedule(year: int) -> pd.DataFrame:
    """Récupère le calendrier d'une saison, avec cache local sur disque.
 
    Le cache FastF1 standard ne couvre pas toujours cet endpoint efficacement,
    donc on le sauvegarde nous-mêmes pour ne jamais le re-fetch inutilement.
    """
    cache_file = os.path.join(SCHEDULE_CACHE_DIR, f"{year}.csv")
    if os.path.exists(cache_file):
        return pd.read_csv(cache_file)
 
    try:
        schedule = fastf1.get_event_schedule(year, include_testing=False)
    except Exception as e:
        if _is_rate_limit_error(e):
            raise RateLimitError(str(e)) from e
        raise
 
    schedule.to_csv(cache_file, index=False)
    return schedule
 
# Réduit le bruit des logs FastF1 (passer à DEBUG en cas de souci réseau)
logging.getLogger('fastf1').setLevel(logging.WARNING)

apping circuit -> code à 3 lettres (clé = session.event['Location'])

In [ ]:
LOCATION_CODES = {
    'Sakhir': 'BHR',
    'Jeddah': 'SAU',
    'Melbourne': 'AUS',
    'Imola': 'EMI',
    'Miami': 'MIA',
    'Montmeló': 'ESP', 'Barcelona': 'ESP',
    'Monaco': 'MON',
    'Baku': 'AZE', # Azerabijan
    'Montréal': 'CAN', 'Montreal': 'CAN',
    'Spielberg': 'AUT',
    'Silverstone': 'GBR',
    'Le Castellet': 'FRA',
    'Budapest': 'HUN',
    'Spa-Francorchamps': 'BEL',
    'Zandvoort': 'NED',
    'Monza': 'ITA',
    'Marina Bay': 'SIN', 'Singapore': 'SIN',
    'Suzuka': 'JPN',
    'Lusail': 'QAT',
    'Austin': 'USA',
    'Mexico City': 'MEX',
    'São Paulo': 'BRA', 'Sao Paulo': 'BRA',
    'Las Vegas': 'LAS',
    'Yas Island': 'ABU', 'Abu Dhabi': 'ABU',
    'Shanghai': 'CHN',
    'Portimão': 'POR', 'Portimao': 'POR',
}
 
_used_codes = set()
 
 
def get_race_code(location: str) -> str:
    """Renvoie un code 3 lettres unique pour un circuit donné."""
    if location in LOCATION_CODES:
        return LOCATION_CODES[location]
    # Fallback : génère un code à partir du nom si non répertorié
    base_fallback = location[:3].upper()
    fallback = base_fallback
    suffix = 1
    while fallback in _used_codes:
        fallback = f"{base_fallback}{suffix}"
        suffix += 1
    print(f" Circuit inconnu '{location}', code généré automatiquement: {fallback}")
    return fallback
 
 
def build_race_id(year: int, location: str) -> str:
    code = get_race_code(location)
    return f"{code}{str(year)[-2:]}"

Extraction d'une course


In [10]:
def process_race(year: int, round_number: int, event_name: str, location: str):
    # Le RaceID peut être calculé à partir du calendrier seul, sans charger
    # la session -> on peut donc vérifier si ce fichier existe déjà AVANT
    # de consommer un appel API, ce qui rend le pipeline reprenable
    # gratuitement après une limite de quota.
    race_id = build_race_id(year, location)
    out_path = os.path.join(INDIVIDUAL_DIR, f"{race_id}.csv")
    if os.path.exists(out_path):
        print(f" {race_id} déjà extrait, ignoré.")
        return pd.read_csv(out_path)
 
    print(f"\n {year} - Round {round_number}: {event_name}")
    try:
        session = fastf1.get_session(year, round_number, SESSION_TYPE)
        session.load(telemetry=True, laps=True, weather=True)
    except Exception as e:
        if _is_rate_limit_error(e):
            raise RateLimitError(str(e)) from e
        print(f" Impossible de charger la session: {e}")
        return None
 
    if session.laps is None or session.laps.empty:
        print("Aucun tour disponible pour cette session, ignorée.")
        return None
 
    laps = session.laps.copy()
    laps['RaceID'] = race_id
    laps['Year'] = year
    laps['RaceName'] = session.event['EventName']
    laps['Circuit'] = location
 
    # Fusion météo
    try:
        weather_df = session.weather_data[['Time', 'AirTemp', 'TrackTemp', 'Rainfall']]
        laps = pd.merge_asof(
            laps.sort_values('Time'),
            weather_df.sort_values('Time'),
            on='Time', direction='nearest'
        )
    except Exception as e:
        print(f" Météo non disponible: {e}")
        laps['AirTemp'], laps['TrackTemp'], laps['Rainfall'] = pd.NA, pd.NA, pd.NA
 
    laps['LapTime_Seconds'] = laps['LapTime'].dt.total_seconds()
    laps['TireAge'] = laps['TyreLife']
 
    final = laps[[
        'RaceID', 'Year', 'RaceName', 'Circuit', 'Driver', 'Team', 'LapNumber',
        'Position', 'LapTime_Seconds', 'Compound', 'TireAge', 'Stint',
        'PitOutTime', 'PitInTime', 'AirTemp', 'TrackTemp', 'Rainfall'
    ]].sort_values(by=['Driver', 'LapNumber']).reset_index(drop=True)
 
    final.to_csv(out_path, index=False)
    print(f" Sauvegardé: {out_path} ({len(final)} lignes)")
    return final

Pipeline principal

In [11]:
def run_pipeline():
    failed = []
    stopped_early = False
 
    for year in YEARS:
        try:
            schedule = get_schedule(year)
        except RateLimitError as e:
            print(f"\n Limite de quota API atteinte en récupérant le calendrier {year}: {e}")
            print("   Arrêt du pipeline. Relancez le script plus tard : les courses déjà")
            print("   extraites seront ignorées automatiquement (reprise gratuite).")
            stopped_early = True
            break
        except Exception as e:
            print(f" Impossible de récupérer le calendrier {year}: {e}")
            continue
 
        for _, event in schedule.iterrows():
            round_number = event['RoundNumber']
            event_name = event['EventName']
            location = event['Location']
            try:
                df = process_race(year, round_number, event_name, location)
            except RateLimitError as e:
                print(f"\n Limite de quota API atteinte sur '{event_name}' ({year}): {e}")
                print("   Arrêt du pipeline. Relancez le script plus tard : les courses déjà")
                print("   extraites seront ignorées automatiquement (reprise gratuite).")
                stopped_early = True
                break
            if df is None:
                failed.append(f"{year} - {event_name}")
        if stopped_early:
            break
 
    # Reconstruit le dataset complet à partir de TOUS les fichiers déjà sur
    # disque, peu importe quand ils ont été extraits. Ainsi le master file
    # est toujours cohérent même si le pipeline s'est arrêté en cours de route.
    race_files = sorted(glob.glob(os.path.join(INDIVIDUAL_DIR, '*.csv')))
    if race_files:
        master = pd.concat((pd.read_csv(f) for f in race_files), ignore_index=True)
        master.to_csv(MASTER_FILE, index=False)
        print(f"\n {len(race_files)} courses disponibles au total dans {INDIVIDUAL_DIR}/")
        print(f" Dataset complet reconstruit: {MASTER_FILE}")
    else:
        print("\n Aucune donnée extraite.")
 
    if failed:
        print(f"\n Courses échouées ({len(failed)}):")
        for f in failed:
            print(f"  - {f}")
 
    if stopped_early:
        print("\n Pipeline interrompu avant la fin. Relancez le script pour continuer.")
 
 
if __name__ == '__main__':
    run_pipeline()

 BHR24 déjà extrait, ignoré.
 SAU24 déjà extrait, ignoré.
 AUS24 déjà extrait, ignoré.
 JPN24 déjà extrait, ignoré.
 CHN24 déjà extrait, ignoré.
 MIA24 déjà extrait, ignoré.
 EMI24 déjà extrait, ignoré.
 MON24 déjà extrait, ignoré.
 CAN24 déjà extrait, ignoré.
 ESP24 déjà extrait, ignoré.
 AUT24 déjà extrait, ignoré.
 GBR24 déjà extrait, ignoré.
 HUN24 déjà extrait, ignoré.
 BEL24 déjà extrait, ignoré.
 NED24 déjà extrait, ignoré.
 ITA24 déjà extrait, ignoré.
 AZE24 déjà extrait, ignoré.
 SIN24 déjà extrait, ignoré.
 USA24 déjà extrait, ignoré.
 MEX24 déjà extrait, ignoré.
 BRA24 déjà extrait, ignoré.
 LAS24 déjà extrait, ignoré.
 QAT24 déjà extrait, ignoré.

 2024 - Round 24: Abu Dhabi Grand Prix
 Sauvegardé: ./Raw/Data/Processed\by_race\ABU24.csv (1035 lignes)

 2025 - Round 1: Australian Grand Prix


core        WARNING 	Fixed incorrect tyre stint information for driver '87'
core        WARNING 	Fixed incorrect tyre stint information for driver '30'
core        WARNING 	Fixed incorrect tyre stint information for driver '5'
core        WARNING 	Driver 4 completed the race distance 00:00.022000 before the recorded end of the session.
_api        WARNING 	Driver 241: Position data is incomplete!
_api        WARNING 	Driver 242: Position data is incomplete!
_api        WARNING 	Driver 243: Position data is incomplete!


 Sauvegardé: ./Raw/Data/Processed\by_race\AUS25.csv (927 lignes)

 2025 - Round 2: Chinese Grand Prix
 Sauvegardé: ./Raw/Data/Processed\by_race\CHN25.csv (1065 lignes)

 2025 - Round 3: Japanese Grand Prix
 Sauvegardé: ./Raw/Data/Processed\by_race\JPN25.csv (1059 lignes)

 2025 - Round 4: Bahrain Grand Prix


_api        WARNING 	Driver  2: Car data is incomplete!
_api        WARNING 	Driver  3: Car data is incomplete!
_api        WARNING 	Driver  8: Car data is incomplete!
_api        WARNING 	Driver  9: Car data is incomplete!
_api        WARNING 	Driver 11: Car data is incomplete!
_api        WARNING 	Driver 15: Car data is incomplete!
_api        WARNING 	Driver 17: Car data is incomplete!
_api        WARNING 	Driver 19: Car data is incomplete!
_api        WARNING 	Driver 20: Car data is incomplete!
_api        WARNING 	Driver 21: Car data is incomplete!
_api        WARNING 	Driver 24: Car data is incomplete!
_api        WARNING 	Driver 25: Car data is incomplete!
_api        WARNING 	Driver 26: Car data is incomplete!
_api        WARNING 	Driver 28: Car data is incomplete!
_api        WARNING 	Driver 29: Car data is incomplete!
_api        WARNING 	Driver 44: Car data is incomplete!
_api        WARNING 	Driver 55: Car data is incomplete!
_api        WARNING 	Driver 63: Car data is inco

 Sauvegardé: ./Raw/Data/Processed\by_race\BHR25.csv (1128 lignes)

 2025 - Round 5: Saudi Arabian Grand Prix


...

 Sauvegardé: ./Raw/Data/Processed\by_race\BRA25.csv (1251 lignes)

 2025 - Round 22: Las Vegas Grand Prix


_api        WARNING 	Failed to align laps for drivers: ['5']
core        WARNING 	Fixed incorrect tyre stint information for driver '63'
_api        WARNING 	Driver 11: Car data is incomplete!
_api        WARNING 	Driver 20: Car data is incomplete!
_api        WARNING 	Driver 24: Car data is incomplete!
_api        WARNING 	Driver 77: Car data is incomplete!
_api        WARNING 	Driver  5: Car data is incomplete!
_api        WARNING 	Driver  6: Car data is incomplete!
_api        WARNING 	Driver 12: Car data is incomplete!
_api        WARNING 	Driver 87: Car data is incomplete!
_api        WARNING 	Driver 11: Position data is incomplete!
_api        WARNING 	Driver 20: Position data is incomplete!
_api        WARNING 	Driver 24: Position data is incomplete!
_api        WARNING 	Driver 77: Position data is incomplete!
_api        WARNING 	Driver  5: Position data is incomplete!
_api        WARNING 	Driver  6: Position data is incomplete!
_api        WARNING 	Driver 12: Position data is 

 Sauvegardé: ./Raw/Data/Processed\by_race\LAS25.csv (886 lignes)

 2025 - Round 23: Qatar Grand Prix


logger      WARNING 	Failed to add first lap time from Ergast!
_api        WARNING 	Driver 241: Position data is incomplete!
_api        WARNING 	Driver 242: Position data is incomplete!
_api        WARNING 	Driver 243: Position data is incomplete!


 Sauvegardé: ./Raw/Data/Processed\by_race\QAT25.csv (1067 lignes)

 2025 - Round 24: Abu Dhabi Grand Prix
 Sauvegardé: ./Raw/Data/Processed\by_race\ABU25.csv (1156 lignes)

 90 courses disponibles au total dans ./Raw/Data/Processed\by_race/
 Dataset complet reconstruit: ./Raw/Data/Processed\all_races_2022_2025.csv
